In [ ]:
import sys, os
import pickle

import numpy as np
import pandas as pd
from scipy import sparse
import scanpy as sc

import gseapy as gp

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from adjustText import adjust_text
import textwrap

from natsort import natsorted
from glob import glob
from tqdm import tqdm

In [ ]:
sys.path.append('../auxiliary_scripts')
from graphics import dotplot
from io import read_gmt, write_gmt

# receiver label enrichment

In [ ]:
receiver_mageck_outdir = '../results/receiver_mageck'
os.makedirs(receiver_mageck_outdir, exist_ok=True)

In [ ]:
nb_est = pd.read_table('../results/label_matrix_SCT_counts_nbinom_est.txt', index_col=0).T
nb_est

In [ ]:
# calculate size factors using cancer fraction
cancer = nb_est.loc[:, nb_est.columns.str.endswith('cancer')]

cancer_mean = cancer.mean(axis=1)

mean_sf = (nb_est / cancer_mean.values[:, None]).mean() # size factor
mean_norm = (nb_est / mean_sf).replace(np.inf, np.nan)
mean_norm = mean_norm.dropna(how='all', axis=1)

In [ ]:
safes = mean_norm.loc[mean_norm.index.str.startswith('safe')]
safes

In [ ]:
zscore = (mean_norm - safes.mean()) / safes.std().replace(0, 1)
zscore = zscore.loc[:, zscore.columns.str.endswith(tuple(valid_celltypes))]
zscore

In [ ]:
tumors = zscore.columns.str.split(':').str[0].unique()
tumors

In [ ]:
# within tumor normalize to cancer zscores
stack = []
for t in tumors:
    local = zscore.loc[:, zscore.columns.str.startswith(t+':')]
    cancer = local.loc[:, local.columns.str.endswith(':cancer')]
    norm = local - cancer.values
    norm = norm.drop(cancer.columns, axis=1)
    stack.append(norm)
    
dzscore = pd.concat(stack, axis=1)
dzscore

In [ ]:
# write
dzscore.to_csv(os.path.join(receiver_mageck_outdir, 'delta_zscore_vs_cancer_within_tumor.txt'), sep='\t')

In [ ]:
celltypes = dzscore.columns.str.split(':').str[-1].unique()
celltypes

In [ ]:
print('echo safe > safe.txt')

rankings = {}
for ct in celltypes:
    if ct == 'cancer':
        continue

    local = dzscore.loc[:, dzscore.columns.str.endswith(':'+ct)]
    local = local.melt(ignore_index=False)
    local.index = local.index + '_' + local['variable'].str.split(':').str[0]
    
    # mean dzscore across tumors for ranking statistic
    local_agg = local.groupby(local.index.str.split('_').str[:-1].str.join('_'))['value'].mean().to_frame()
    local_agg['symbol'] = local_agg.index.str.split('_').str[0]
    local_agg['pool'] = 'list'
    local_agg['p.high'] = -local_agg['value']
    local_agg['p.low'] = local_agg['value']
    local_agg['prob'] = 1
    local_agg['chosen'] = 1
    rankings[ct] = local_agg.copy()
    
    # write
    local_agg[['symbol', 'pool', 'p.high', 'prob', 'chosen']].sort_values('p.high').to_csv(os.path.join(receiver_mageck_outdir, ct+'.phigh.txt'), sep='\t')
    local_agg[['symbol', 'pool', 'p.low', 'prob', 'chosen']].sort_values('p.low').to_csv(os.path.join(receiver_mageck_outdir, ct+'.plow.txt'), sep='\t')
    
    safe_sgrna = local_agg.index[local_agg.index.str.startswith('safe_')]
    pd.Series(safe_sgrna).to_csv(os.path.join(receiver_mageck_outdir, ct+'.safe_ids.txt'), index=False, header=None)


In [ ]:
# calculate average effect size and collate into gene_summary.txt file

for k in rankings.keys():
    outpath = os.path.join(receiver_mageck_outdir, k+'.gene_summary.txt')
    high = pd.read_table(os.path.join(receiver_mageck_outdir, k+'.gene.high.txt'), index_col=0)
    low = pd.read_table(os.path.join(receiver_mageck_outdir, k+'.gene.low.txt'), index_col=0)
    
    local = custom_rankings[k]
    
    median_effect = local.groupby('symbol')['value'].median()
    mean_effect = local.groupby('symbol')['value'].mean()
    
    high.columns = ['num', 'score', 'p-value', 'fdr', 'goodsgrna']
    high['rank'] = np.arange(1, len(high)+1)
    high.columns = 'pos|' + high.columns
    
    low.columns = ['num', 'score', 'p-value', 'fdr', 'goodsgrna']
    low['rank'] = np.arange(1, len(low)+1)
    low.columns = 'neg|' + low.columns
    
    gene_summary = pd.concat([low, high], axis=1)
    gene_summary['median_effect'] = gene_summary.index.map(median_effect)
    gene_summary['mean_effect'] = gene_summary.index.map(mean_effect)
    gene_summary.index.name = 'id'
    
    gene_summary.to_csv(outpath, sep='\t')

# receiver DEG

In [ ]:
# convert adata_proc_singlets.h5ad to R object (qs2 for fast I/O)
!Rscript ../auxiliary_scripts/adata2seurat.R ../results/adata_proc_singlets.h5ad

In [ ]:
# run for all cell types
!Rscript ../auxiliary_scripts/receiver_effects.R ../results/adata_proc_singlets_seurat.qs2 ../results/label_matrix_SCT_counts.txt major_celltype:CD4
!Rscript ../auxiliary_scripts/receiver_effects.R ../results/adata_proc_singlets_seurat.qs2 ../results/label_matrix_SCT_counts.txt major_celltype:CD8
!Rscript ../auxiliary_scripts/receiver_effects.R ../results/adata_proc_singlets_seurat.qs2 ../results/label_matrix_SCT_counts.txt major_celltype:NK
!Rscript ../auxiliary_scripts/receiver_effects.R ../results/adata_proc_singlets_seurat.qs2 ../results/label_matrix_SCT_counts.txt major_celltype:Treg
!Rscript ../auxiliary_scripts/receiver_effects.R ../results/adata_proc_singlets_seurat.qs2 ../results/label_matrix_SCT_counts.txt major_celltype:macrophage
!Rscript ../auxiliary_scripts/receiver_effects.R ../results/adata_proc_singlets_seurat.qs2 ../results/label_matrix_SCT_counts.txt major_celltype:monocyte
!Rscript ../auxiliary_scripts/receiver_effects.R ../results/adata_proc_singlets_seurat.qs2 ../results/label_matrix_SCT_counts.txt major_celltype:MDSC
!Rscript ../auxiliary_scripts/receiver_effects.R ../results/adata_proc_singlets_seurat.qs2 ../results/label_matrix_SCT_counts.txt major_celltype:neutrophil
!Rscript ../auxiliary_scripts/receiver_effects.R ../results/adata_proc_singlets_seurat.qs2 ../results/label_matrix_SCT_counts.txt major_celltype:cDC1
!Rscript ../auxiliary_scripts/receiver_effects.R ../results/adata_proc_singlets_seurat.qs2 ../results/label_matrix_SCT_counts.txt major_celltype:cDC2
!Rscript ../auxiliary_scripts/receiver_effects.R ../results/adata_proc_singlets_seurat.qs2 ../results/label_matrix_SCT_counts.txt major_celltype:MoDC

In [ ]:
!mkdir ../results/receiver_effects
!mv -t ../results/receiver_effects ../results/*_vs_safe.glm_gp_contrast.txt

# gene set selection

In [ ]:
from kneed import KneeLocator

def find_knee(rnk, verbose=False):
    rnk = rnk.sort_values()
    
    zidx = np.where(rnk == 0)[0]
    if len(zidx) == 0:
        idx = np.argmin(rnk.abs())
        zidx = [idx, idx]

    # negative
    y = rnk.values[:zidx[0]]  # first zero value
    x = np.arange(len(y))
    kneedle = KneeLocator(x, y, S=1.0, curve='concave', direction='increasing')
    n1 = kneedle.knee

    # positive
    y = rnk.values[zidx[-1]:] # last zero value
    x = np.arange(zidx[-1], len(rnk))
    kneedle = KneeLocator(x, y, S=1.0, curve='convex', direction='increasing')
    n2 = kneedle.knee

    dn_genes = rnk.index[:n1].tolist()
    up_genes = rnk.index[n2:].tolist()
    if verbose: print(len(dn_genes), len(up_genes))
    return dn_genes, up_genes
    

In [ ]:
immune_deg_resdir = '../results/receiver_effects'
immune_deg_files = natsorted(glob(os.path.join(immune_deg_resdir, '*glm_gp_contrast.txt')))

immune_degs = []
for f in tqdm(immune_deg_files):
    _, celltype, target = os.path.basename(f).split('_vs_safe')[0].split('--')[-1].split('.')
    local = pd.read_table(f, index_col=0)
    local['target'] = target
    local['celltype'] = celltype
    immune_degs.append(local.copy())
    
immune_degs = pd.concat(immune_degs)
immune_degs['slogpadj'] = np.sign(immune_degs['lfc']) * -np.log10(immune_degs['adj_pval'])
immune_degs['rnk_stat'] = np.sign(immune_degs['lfc']) * immune_degs['f_statistic']
immune_degs

In [ ]:
immune_knee_genes = {}

for group in tqdm(immune_degs.groupby(['target', 'celltype'])):
    (t, c), local = group
    local = local.set_index('name').copy()
    dn, up = find_knee(local['rnk_stat'].sort_values())
    up = up[::-1]
    
    immune_knee_genes[f'{c}::{t}_up'] = up
    immune_knee_genes[f'{c}::{t}_down'] = dn
len(immune_knee_genes)

In [ ]:
# write
write_gmt(immune_knee_genes, '../results/immune_kneedle.gmt')

# AUCell scoring

In [ ]:
%%R
library(qs2)
library(Seurat)
library(AUCell)

gmt_file <- '../results/immune_kneedle.gmt'
qs2_file <- '../results/adata_proc_singlets_seurat.qs2'
outdir   <- '../results/AUCell_results'
n_cores  <- 1

dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

cat("Parsing GMT...\n")
lines <- readLines(gmt_file)
all_genesets <- lapply(lines, function(l) {
  parts <- strsplit(l, "\t")[[1]]
  list(name = parts[1], genes = parts[-(1:2)])
})
names(all_genesets) <- sapply(all_genesets, `[[`, "name")
all_genesets <- lapply(all_genesets, `[[`, "genes")

celltypes <- unique(sub("::.*", "", names(all_genesets)))
cat(sprintf("Found %d gene sets across %d celltypes\n", length(all_genesets), length(celltypes)))

cat("Loading Seurat object...\n")
seurat_obj <- qs_read(qs2_file)

cat("Extracting counts matrix...\n")
all_counts <- tryCatch(
  GetAssayData(seurat_obj, layer = "counts", assay = "RNA"),
  error = function(e) GetAssayData(seurat_obj, slot = "counts", assay = "RNA")
)
cat(sprintf("Counts matrix: %d genes x %d cells\n", nrow(all_counts), ncol(all_counts)))

meta <- seurat_obj@meta.data

for (ct in celltypes) {
  cat(sprintf("\n=== %s ===\n", ct))

  cells <- rownames(meta)[meta$major_celltype == ct]
  if (length(cells) == 0) {
    cat("  No cells found, skipping\n")
    next
  }
  cat(sprintf("  %d cells\n", length(cells)))

  ct_pattern <- paste0("^", ct, "::")
  ct_names   <- grep(ct_pattern, names(all_genesets), value = TRUE)
  if (length(ct_names) == 0) {
    cat("  No gene sets found, skipping\n")
    next
  }
  ct_genesets <- all_genesets[ct_names]
  names(ct_genesets) <- sub(ct_pattern, "", ct_names)
  cat(sprintf("  %d gene sets\n", length(ct_genesets)))

  ct_counts <- all_counts[, cells, drop = FALSE]

  cat("  Building rankings...\n")
  rankings <- AUCell_buildRankings(ct_counts, nCores = n_cores, plotStats = FALSE)

  cat("  Calculating AUC...\n")
  auc_max <- ceiling(0.05 * nrow(ct_counts))
  auc_res <- AUCell_calcAUC(ct_genesets, rankings, aucMaxRank = auc_max, nCores = n_cores)

  scores <- t(getAUC(auc_res))  # n_cells x n_genesets

  outfile <- file.path(outdir, paste0(ct, "_AUCell_scores.csv"))
  write.csv(scores, outfile)
  cat(sprintf("  Saved: %s  [%d x %d]\n", outfile, nrow(scores), ncol(scores)))
}

In [ ]:
infiles = natsorted(glob('../results/AUCell_results/*_AUCell_scores.csv'))
len(infiles)

In [ ]:
stack = []
delta = []
zdelta = []
for f in infiles:
    name = os.path.basename(f).split('_')[0]
    print(name)
    
    local = pd.read_csv(f, index_col=0)
    stack.append(local.copy())
    
    up = local.loc[:, local.columns.str.endswith('_up')].copy()
    up.columns = up.columns.str.split('_').str[0]
    dn = local.loc[:, local.columns.str.endswith('_down')].copy()
    dn.columns = dn.columns.str.split('_').str[0]
    assert all(up.columns == dn.columns)
    d = up - dn

    
aucell = pd.concat(stack)
delta = pd.concat(delta)

In [ ]:
delta['celltype'] = delta.index.map(adata.obs['fine_celltype'])

delta_median = delta.groupby('celltype').median().dropna(how='all', axis=0)

delta.drop('celltype', axis=1, inplace=True)

In [ ]:
# zscore median values within a cell type
valid_types = [
    'CD8', 'CD4', 'NK',
    'macrophage', 'monocyte'
]

zmat_dict = {}
for c in valid_celltypes:
    local_group = adata.obs[adata.obs['major_celltype'] == c]['fine_celltype'].unique()
    idx = delta_median.index.isin(local_group)
    
    local = delta_median.loc[idx].copy()
    
    # z-score within cell type
    out = (local - local.mean()) / local.std()

    zmat_dict[c] = out.T
    
zmat = pd.concat(zmat_dict.values(), axis=1)

# UMAP embedding

In [ ]:
from umap import UMAP

reducer = UMAP(
    metric='cosine',
    n_neighbors=10, 
    min_dist=0.25, 
    low_memory=False, 
    negative_sample_rate=10,
    n_components=2, 
    random_state=1, 
    verbose=True, 
    n_epochs=10000,
     # local_connectivity = 30,
    learning_rate = 1)

In [ ]:
## z-scored within celltype
input_mat = zmat.copy()

input_mat = input_mat.loc[:, ~input_mat.columns.str.startswith('cancer-')]  # remove cancer
input_mat

In [ ]:
X_umap = reducer.fit_transform(input_mat.fillna(0))

In [ ]:
umap_df = pd.DataFrame(X_umap, index=input_mat.index)

umap_df

In [ ]:
ncols = 5
figscale = 2

# CD4, CD8, NK, macrophage, monocyte
groups = input_mat.columns
groups = [_ for _ in groups if _.startswith('NK')]
n = len(groups)
nrows = n//ncols + (n%ncols != 0)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*figscale, nrows*figscale))
axes = axes.flatten()

for i, group in enumerate(groups):
    ax = axes[i]
    
    c = input_mat[group]
    order = np.argsort(c)
    
    ax.scatter(umap_df[0].iloc[order], umap_df[1].iloc[order], c=input_mat[group].iloc[order], s=40, edgecolor='none')
    
    wrapped_title = "\n".join(textwrap.wrap(group, width=15))
    ax.set_title(wrapped_title)
    ax.set_xticks([])
    ax.set_yticks([])
#     break
    
for j in range(i+1, len(axes)):
    axes[j].axis('off')
    
plt.tight_layout()
plt.show()

# correlation analysis with cancer programs

In [ ]:
def cross_correlation(X, Y):
    """
    X: (n, i) matrix
    Y: (n, j) matrix
    Returns: (i, j) correlation matrix
    """
    X_centered = X - np.mean(X, axis=0)
    Y_centered = Y - np.mean(Y, axis=0)
    
    X_norms = np.sqrt(np.sum(X_centered**2, axis=0))
    Y_norms = np.sqrt(np.sum(Y_centered**2, axis=0))
    
    # (i, n) @ (n, j) -> (i, j)
    corr_matrix = (X_centered.T @ Y_centered) / np.outer(X_norms, Y_norms)
    
    return corr_matrix

In [ ]:
cancer_nmf_ols = pd.read_table('../results/cNMF_cancer/k15_OLS_factor_beta.txt', sep='\t')
cancer_nmf_ols.columns = 'cancer_' + cancer_nmf_ols.columns.astype(str)

In [ ]:
A = cancer_nmf_ols.fillna(0)
B = delta_median.loc[~delta_median.index.str.startswith(('cancer', 'pDC', 'neutrophil')), 
                    A.index].T.fillna(0).copy()

B.columns = ['DC-'+_  if _.startswith(('MoDC', 'cDC')) else _ for _ in B.columns]
B.columns = ['NK-'+_  if _.startswith(('ILC1',)) else _ for _ in B.columns]
B.columns = ['monocyte-'+_  if _.startswith(('MDSC',)) else _ for _ in B.columns]

cross_corr = cross_correlation(A.values, B.values)
cross_corr = pd.DataFrame(cross_corr, index=A.columns, columns=B.columns)

cross_corr

In [ ]:
import matplotlib.patches as mpatches

pdf = cross_corr.copy().fillna(0)

groups = pdf.columns.str.split('-').str[0]
all_celltypes = groups.unique()
lut = {k: cc.glasbey_hv[i] for i, k in enumerate(all_celltypes)}
col_colors = [lut[g] for g in groups]

g = sns.clustermap(pdf, 
                   figsize=(7, 5),
#                    yticklabels=True,
                   col_colors=col_colors,
                   method='ward',
                   center=0, cmap='seismic')

handles = [mpatches.Patch(color=c, label=l) for l, c in lut.items()]

g.fig.legend(
    handles=handles, 
    title='Immune cell type', 
    loc='lower left', 
    bbox_to_anchor=(0.02, 0.02),
    frameon=True,
    fontsize=6, ncol=2
)

plt.show()

# receiver DEG cNMF
run with NVIDIA L4 GPU on Google Cloud instance

In [ ]:
import os, sys

import torch
import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns

import pickle
from glob import glob
from natsort import natsorted
from copy import deepcopy
from tqdm import tqdm

In [ ]:
sys.path.append('../auxiliary_scripts/NMF_gpu')
from NMF.nmf import StandardNMF
from NMF.sweep import run_rank_sweep
from NMF.graphics import plot_pairwise_component_distances, plot_all_clustered_components, plot_sweep_metrics

In [ ]:
def reconst_pos_neg(loadings, scale=True):
    loadings_reconst = np.zeros((len(loadings)//2, loadings.shape[1]))

    pos_tmp = loadings.loc[loadings.index.str.startswith('pos::')]
    neg_tmp = loadings.loc[loadings.index.str.startswith('neg::')]
        
    pos_idx = np.where(pos_tmp.values > neg_tmp.values)
    neg_idx = np.where(neg_tmp.values > pos_tmp.values)
    
    if scale:
        pos_tmp = pos_tmp / pos_tmp.max()
        neg_tmp = neg_tmp / neg_tmp.max()

    loadings_reconst[pos_idx] = pos_tmp.values[pos_idx]
    loadings_reconst[neg_idx] = -neg_tmp.values[neg_idx]

    loadings_reconst = pd.DataFrame(loadings_reconst, index=pos_tmp.index.str.split("::").str[-1])
    loadings_reconst.index.name = None
    
    return loadings_reconst

In [ ]:
targets = np.array(list(immune_degs.groupby('target').groups.keys()))
celltypes = np.array(list(immune_degs.groupby('celltype').groups.keys()))

len(celltypes), len(targets)

In [ ]:
# rows are features
# columns are samples

immune_slogf = immune_degs.pivot(index=['name', 'celltype'], columns='target', values='slogpadj').fillna(0)
immune_slogf.shape

In [ ]:
nmf_outdir = '../results/cNMF_receiver'
os.makedirs(nmf_outdir, exist_ok=True)


In [ ]:
# set matrices to measured genes for each cell type

immune_matrices = {}

for ct in celltypes:
    immune_mat = immune_slogf.xs(ct, level=1)
    
    pos_idx = np.where(immune_mat.values > 0)
    neg_idx = np.where(immune_mat.values < 0)
    
    pos_immune = np.zeros_like(immune_mat)
    neg_immune = np.zeros_like(immune_mat)
    
    pos_immune[pos_idx] = immune_mat.values[pos_idx]
    neg_immune[neg_idx] = np.abs(immune_mat.values[neg_idx])
    
    split_immune = pd.DataFrame(np.vstack([pos_immune, neg_immune]),
                               columns=immune_mat.columns, 
                               index=list('pos::' + immune_mat.index) + list('neg::' + immune_mat.index))

    # check alignment
    print(ct, split_immune.shape)
    
    immune_matrices[ct] = split_immune

    pd.Series(split_immune.index).to_csv(os.path.join(nmf_outdir, f'{ct}_genes.txt'), header=None, index=False)
    np.save(os.path.join(nmf_outdir, f'{ct}_input_matrix.npy'), split_immune.values)

In [ ]:
# perform rank sweep on all cell types

rank_range = np.arange(2, 21)

for ct in celltypes:
    print(ct)
    outpath = os.path.join(nmf_outdir, f'{ct}_rank_sweep_results.pkl')
    if os.path.exists(outpath):
        print('output exists, skipping')
        continue

    local = immune_matrices[ct].copy()
    
    local_rank_sweep = run_rank_sweep(local.values, 
                                        model_cls=StandardNMF,
                                        rank_range=rank_range, 
                                        n_runs=100, 
                                        device=device)

    with open(outpath, 'wb') as h:
        pickle.dump(local_rank_sweep, h)

In [ ]:
c = 'CD8'

with open(os.path.join(nmf_outdir, f'{c}_rank_sweep_results.pkl'), 'rb') as f:
    rank_sweep_results = pickle.load(f)

In [ ]:
plot_sweep_metrics(rank_sweep_results,  
                   right_key='H_ccc')

In [ ]:
ranks = pd.read_table('../receiver_NMF_ranks.txt').squeeze()

In [ ]:
for c in tqdm(ranks.index):
    k = ranks.loc[c]    
    genes = pd.read_table(os.path.join(nmf_outdir, f'{c}_genes.txt'), header=None, sep='\t').squeeze().values

    with open(os.path.join(nmf_outdir, f'{c}_rank_sweep_results.pkl'), 'rb') as h:
        rank_results = pickle.load(h)
            
    input_matrix = np.load(os.path.join(basedir, f'{c}_input_matrix.npy'))
    tmp = pd.DataFrame(input_matrix, index=genes, columns=targets)
    input_reconst = reconst_pos_neg(tmp, scale=False)
        
    gene_loadings = pd.DataFrame(rank_results[k]['W'], index=genes)
    gene_loadings.index.name = None
    factor_loadings = pd.DataFrame(rank_results[k]['H'].T, index=targets)
    factor_loadings.index.name = None
    
    gene_loadings_reconst = reconst_pos_neg(gene_loadings, scale=False)
    
    gene_loadings.to_csv(os.path.join(nmf_outdir, f'{c}_k{k}_gene_loadings.txt'), sep='\t')
    factor_loadings.to_csv(os.path.join(nmf_outdir, f'{c}_k{k}_factor_loadings.txt'), sep='\t')
    gene_loadings_reconst.to_csv(os.path.join(nmf_outdir, f'{c}_k{k}_gene_loadings_reconst.txt'), sep='\t')

# LIA

In [ ]:
# factor loadings
infiles = natsorted(glob(os.path.join(nmf_outdir, '_factor_loadings.txt')))
len(infiles)

In [ ]:
factor_stack = []
for f in infiles:
    f_name = os.path.basename(f).split('_')[0]
    local = pd.read_table(f, index_col=0)
    local.columns = f_name + '_' + local.columns
    factor_stack.append(local_factor)

factor_loadings_mat = pd.concat(factor_stack, axis=1).fillna(0)
factor_loadings_mat_scaled = factor_loadings_mat / factor_loadings_mat.max()

In [ ]:
hallmark = read_gmt('../data/hallmark.v7.0.mm39.gmt')

In [ ]:
# gene loadings
infiles = natsorted(glob(os.path.join(nmf_outdir, '_gene_loadings_reconst.txt')))
len(infiles)

In [ ]:
gs_use = hallmark; geneset_name = 'hallmark'

# save results here
all_res = dict()
nes_res = dict()
fdr_res = dict()

for f in infiles:
    gene_loadings_reconst = pd.read_table(f, index_col=0)
    c = os.path.basename(f).split('_k')[0]  # cell type name
    print(c)
    
    for i in tqdm(gene_loadings_reconst.columns):
        rnk = gene_loadings_reconst[i].sort_values()
        rnk = rnk.loc[rnk!=0]
        pre_res = gp.prerank(rnk=rnk,
                         gene_sets=gs_use,
                         threads=8,
                         min_size=10,
                         max_size=500,
                         permutation_num=1000,
                         outdir=None,
                         seed=0,
                         verbose=False
                        )
        local_res = pre_res.res2d.sort_values('NES').set_index('Term')
        if c not in all_res.keys():
            all_res[c] = {i:local_res.copy()}
            nes_res[c] = {i:local_res['NES'].astype(float)}
            fdr_res[c] = {i:local_res['FDR q-val'].astype(float)}
        else:
            all_res[c][i] = local_res.copy()
            nes_res[c][i] = local_res['NES'].astype(float)
            fdr_res[c][i] = local_res['FDR q-val'].astype(float)

In [ ]:
# gather into dataframes
nes_dfs = {}
fdr_dfs = {}
slogf_dfs = {}

for c in nes_res.keys():
    nes_dfs[c] = pd.concat(nes_res[c], axis=1)
    fdr_dfs[c] = pd.concat(fdr_res[c], axis=1)

    fdrs = fdr_dfs[c].dropna().values.ravel()     
    min_f = fdrs[fdrs!=0].min()

    slogf_dfs[c] = np.sign(nes_dfs[c]) * -np.log10(fdr_dfs[c].clip(lower=min_f))

In [ ]:
gsea_dir = '../results/GSEA'
os.makedirs(gsea_dir, exist_ok=True)
gsea_dir

In [ ]:
# write
for c in tqdm(slogf_dfs.keys()):
    k = ranks.loc[c]
    prefix = os.path.join(gsea_dir, f'{c}_k{k}_')
    
    nes_dfs[c].to_csv(prefix+'NES.txt', sep='\t')
    fdr_dfs[c].to_csv(prefix+'FDR.txt', sep='\t')
    slogf_dfs[c].to_csv(prefix+'slogFDR.txt', sep='\t')
    pd.concat(all_res[c]).to_csv(prefix+'all_results.txt', sep='\t')


In [ ]:
stack = []
for c in slogf_dfs.keys():
    local = slogf_dfs[c].copy()
    local.columns = c + '_' + local.columns
    stack.append(local)

gsea_mat = pd.concat(local, axis=1).fillna(0)

In [ ]:
inflam_pathways = [
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB', 
    'HALLMARK_INTERFERON_GAMMA_RESPONSE',
    'HALLMARK_INFLAMMATORY_RESPONSE',
    'HALLMARK_ALLOGRAFT_REJECTION',
]

LIA_scores = gsea_mat.loc[inflam_pathways].mean(axis=0).sort_values(ascending=False)

In [ ]:
top_LIA_scores = LIA_scores.reset_index()
top_LIA_scores['celltype'] = top_LIA_scores['index'].str.split('_').str[0]
top_LIA_scores = top_LIA_scores.drop_duplicates('celltype', keep='first')
top_LIA_scores = top_LIA_scores.set_index('index')
top_LIA_scores

In [ ]:
whitelist_celltypes = pd.Index([
    'CD4', 
    'CD8', 
    'NK',
    'macrophage', 
    'monocyte', 
    'cDC1', 'cDC2', 'MoDC',
])

In [ ]:
tmp = gsea_mat.loc[:, top_LIA_scores.index]
tmp = tmp.loc[tmp.abs().max(axis=1) > -np.log10(0.05), tmp.columns.str.startswith(tuple(whitelist_celltypes + '_'))]
tmp.index = tmp.index.str.split('HALLMARK_').str[-1].str.replace('_', ' ').str.capitalize()

sns.clustermap(tmp,
               yticklabels=True,
               col_cluster=True,
               figsize=(5, 8), 
               dendrogram_ratio=(0.2, 0.1), 
               cbar_pos=(0.55, 0.12, 0.12, 0.03),
               cbar_kws={"orientation": "horizontal"},
               cmap='RdBu_r', center=0, method='ward')

plt.show()

In [ ]:
picked_factors = top_LIA_scores.index[top_LIA_scores.index.str.startswith(tuple(whitelist_celltypes+'_'))]
LIA_loadings_mean = factor_loadings_mat_scaled.loc[:, picked_factors].mean(axis=1)

In [ ]:
LIA_loadings_mat = factor_loadings_mat_scaled.loc[:, picked_factors].T

LIA_loadings_mat['celltype'] = LIA_loadings_mat.index.str.split('_').str[0]
LIA_loadings_mat = LIA_loadings_mat.groupby('celltype').mean().T

LIA_loadings_mat = LIA_loadings_mat / nmf_factor_loading_score.max()
LIA_loadings_mat = LIA_loadings_mat.loc[targets, whitelist_celltypes]

In [ ]:
# load cancer cell fitness data
fitness = pd.read_table('../results/mageck/invivo_v_input.gene_summary.txt', index_col=0)
assert (fitness['pos|lfc'] == fitness['neg|lfc']).all()
fitness['lfc'] = fitness['pos|lfc']

fitness

In [ ]:
x_metric = 'lfc'

x = fitness[x_metric]
y = LIA_loadings_mean.reindex(fitness.index)
s = (LIA_loadings_mat > 0.2).sum(axis=1).reindex(fitness.index)
c = LIA_loadings_mat.max(axis=1).reindex(fitness.index); color_method='max'

def size_transform(s):
    return (s+1)**2 * 40

p = plt.scatter(x, y, 
                s=size_transform(s), 
                cmap='Reds', c=c,
                edgecolor='k', lw=0.5)
ax = plt.gca()

# add text labels
texts = []
to_plot = y[y>y.std()*2].sort_values(ascending=False).index.tolist()
to_plot += ['Mif']
for i in to_plot:
    texts.append(plt.text(x.loc[i], y.loc[i], i, c='k'))
adjust_text(texts, x=x, y=y, arrowprops=dict(arrowstyle='-', color='k'), force_text=(0.5, 0.5))

### add legends
# dot size legend
size_legend_vals = np.array([1, 2, 3])

legend_handles = [
    plt.scatter([], [], marker="o", 
                color="lightgrey", 
                edgecolor="black", lw=0.5,
        s=size_transform(val),
        label=val)
    for val in size_legend_vals
]

size_legend = ax.legend(
    handles=legend_handles, title="Num cell types",
    loc="upper left",
    labelspacing=1, fontsize=9, title_fontsize=10
)
ax.add_artist(size_legend)

# colorbar
if color_method in ('sum', 'max'):
    axins = ax.inset_axes([0.25, 0.75, 0.03, 0.15])
    cbar = plt.colorbar(p, cax=axins, orientation="vertical")
    cbar.ax.set_title(color_method)


# dot color legend
if color_method == 'celltype':
    handles = [
        plt.scatter([], [], marker='o', color=color, s=20, label=label) 
        for label, color in lut.items()
    ]
    ax.legend(handles=handles, title="celltype", loc='center left', ncol=1, bbox_to_anchor=(1, .5))

plt.axvline(0, ls='-', lw=1, color='k')
plt.axvline(-x.std(), ls='--', lw=1, color='grey')
plt.axvline(x.std(), ls='--', lw=1, color='grey')
plt.xlabel(f'cancer fitness {x_metric}')
plt.ylabel('LIA score')

plt.tight_layout()
plt.show()

In [ ]:
tmp = LIA_loadings_mat.copy()
tmp = tmp.loc[(tmp > 0.2).any(axis=1), whitelist_celltypes]  # filter

g_lia = sns.clustermap(tmp.T, 
                   method='ward',
                   figsize=(8, 3),
                   dendrogram_ratio=0.1,
                   xticklabels=True,
                   cmap='Reds')

g_lia.ax_heatmap.tick_params(axis='x', labelsize=7)
y_order = tmp.columns[g_lia.dendrogram_row.reordered_ind]
plt.show()